In [2]:
from rdflib import Graph, Namespace, URIRef, RDF, RDFS, OWL
from rdflib.namespace import OWL as OWL_NS
import pandas as pd
from IPython.display import display

OFL  = Namespace("https://purl.bioontology.org/ontology/OFL/")
OBO  = Namespace("http://purl.obolibrary.org/obo/")

PREFIXES = """
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:     <http://www.w3.org/2002/07/owl#>
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX ofl:     <https://purl.bioontology.org/ontology/OFL/>
PREFIX dcterms: <http://purl.org/dc/terms/>
"""

TBOX = "flap_onto_with_variants_2_inferred.owl"
#ABOX = "ofl_abox_5k_inferred.owl"
#FMA  = "fma_obo.owl"   # set to None to skip (faster load, labels show as IRI)

g = Graph()
# TBox-only graph (inferred TBox + FMA labels, no ABox)
g_tbox = Graph()
g_tbox.parse(TBOX, format="xml")
if FMA:
    g_tbox.parse(FMA, format="xml")
print(f"TBox-only graph: {len(g_tbox):,} triples")

"""g.parse(TBOX, format="xml");  n_tbox = len(g);  print(f"TBox: {n_tbox:,} triples")
g.parse(ABOX, format="xml");  n_abox = len(g) - n_tbox; print(f"ABox: {n_abox:,} triples")
if FMA:
    g.parse(FMA, format="xml"); print(f"FMA:  {len(g)-n_tbox-n_abox:,} triples")
print(f"Total: {len(g):,} triples")"""

TBox-only graph: 677,373 triples


'g.parse(TBOX, format="xml");  n_tbox = len(g);  print(f"TBox: {n_tbox:,} triples")\ng.parse(ABOX, format="xml");  n_abox = len(g) - n_tbox; print(f"ABox: {n_abox:,} triples")\nif FMA:\n    g.parse(FMA, format="xml"); print(f"FMA:  {len(g)-n_tbox-n_abox:,} triples")\nprint(f"Total: {len(g):,} triples")'

In [3]:
def run(sparql, limit=20, title="", graph=None):
    if graph is None:
        graph = g
    result = graph.query(PREFIXES + sparql)
    cols = [str(v) for v in result.vars]
    rows = list(result)
    data = [{c: (str(getattr(r, c)) if getattr(r, c) is not None else "—") for c in cols} for r in rows]
    df = pd.DataFrame(data, columns=cols)
    if title:
        print(f"\n{'='*60}")
        print(title)
    print(f"  {len(rows):,} row(s)" + (f" (showing first {limit})" if len(rows) > limit else ""))
    display(df.head(limit))
    return df

In [15]:
# Query 1 — all perforator flap classes (fast positive query)
set_perf_flaps = {
    (str(r.flap), str(r.flapLabel))
    for r in g_tbox.query(PREFIXES + """
        SELECT DISTINCT ?flap ?flapLabel WHERE {
          ?flap a owl:Class .
          ?flap rdfs:subClassOf* ofl:OFLID10002 .
          ?flap rdfs:label ?flapLabel .
          FILTER(CONTAINS(LCASE(STR(?flapLabel)), "perforator"))
        }
    """)
}

# Query 2 — flap classes that DO have a perforator part (also a positive query)

set_has_part_perf_flaps = { # Set comprehension
    str(r.flap)
    for r in g_tbox.query(PREFIXES + """
        SELECT DISTINCT ?flap WHERE {
          # Look for the 'Ofl part' restriction
          ?flap rdfs:subClassOf [
                      a owl:Restriction ;
                        owl:onProperty ofl:OFLID13296 ; # 'Ofl part'
                        owl:someValuesFrom [
                                                a owl:Restriction ;
                                                  owl:onProperty ofl:OFLID120000 ; # 'perforator vessel of'
                                                  owl:someValuesFrom ?artery
            ]
          ] .
          
        }
    """)
}

# Python set difference — no rdflib negation needed
sorted_lbl_perf_flap = sorted(lbl for iri, lbl in set_perf_flaps)
df_sorted_lbl_perf_flap = pd.DataFrame({'flapLabel': sorted_lbl_perf_flap})
sorted_set_missing = sorted(lbl for iri, lbl in set_perf_flaps if iri not in set_has_part_perf_flaps)
df_missing = pd.DataFrame({'flapLabel': sorted_set_missing})
print(f"\n{'='*60}")
print("TBox (inferred) — perforator flaps missing a perforator vessel as has-part")
print(f"  {len(df_missing)} row(s)")
display(df_sorted_lbl_perf_flap)
display(df_missing)


TBox (inferred) — perforator flaps missing a perforator vessel as has-part
  10 row(s)


,flapLabel
0,Anterior intercostal artery perforator flap
1,Branch-based flap with recognized perforator
2,Deep inferior epigastric perforator flap
3,Deep inferior epigastric perforator flap with ...
4,Deep inferior epigastric perforator flap with ...
5,Deep inferior epigastric perforator flap with ...
6,Direct cutaneous perforator based flaps
7,Direct cutaneous perforator flaps
8,Dorsal intercostal artery perforator flap
9,Dorsal intercostal artery perforator flap with...


,flapLabel
0,Branch-based flap with recognized perforator
1,Direct cutaneous perforator based flaps
2,Direct cutaneous perforator flaps
3,Medial sural artery perforator flap
4,Musculocutaneous perforator based flaps
5,Musculocutaneus perforator flaps
6,Perforator based flaps
7,Septocutaneous perforator based flaps
8,Septocutaneous perforator flaps
9,Type iii: perforator chimerism


In [ ]:
print(set_has_part_perf_flaps)

{'https://purl.bioontology.org/ontology/OFL/OFLID1000343', 'N6e0638c93bca48bab8516dffb74bee38', 'N82e8196e44804bf99bc083485c65ca04', 'N2f7784ad3ae8451ab9e44353e3551b25', 'https://purl.bioontology.org/ontology/OFL/OFLID1000201', 'N6e12ce136f884877be12857ad891629e', 'https://purl.bioontology.org/ontology/OFL/OFLID1000203', 'N9a514821faf24c769bc540d61ae5092f', 'https://purl.bioontology.org/ontology/OFL/OFLID1000365', 'https://purl.bioontology.org/ontology/OFL/OFLID1000370', 'https://purl.bioontology.org/ontology/OFL/OFLID1000367', 'Nf49a3e9f7f6a41bf869176c82e2e61a5', 'https://purl.bioontology.org/ontology/OFL/OFLID1000349', 'https://purl.bioontology.org/ontology/OFL/OFLID10161', 'https://purl.bioontology.org/ontology/OFL/OFLID10204', 'N07b7550ccd7b4c2b9cac92eb89c09a60', 'N0d8bf04d9808483e8819cc1cb6109a70', 'Nb8a399cfeb97451ea79fe44cb7370e18', 'N25cfa555ef914ad8835adaacd1ce3665', 'Nbccd7bb1625e488cae71d7741c17379b', 'N859436038c7e46f3a8a8acf463acf224', 'N004675cec903468cb40f003e0a65cb66', 

In [30]:
run("""
SELECT DISTINCT ?flapLabel ?partLabel
WHERE {
  ?flap rdfs:subClassOf* ofl:OFLID10002 .
  ?flap rdfs:subClassOf ?r .
  ?r a owl:Restriction ; owl:onProperty ofl:OFLID13296 ; owl:someValuesFrom ?filler .
  ?flap rdfs:label ?flapLabel .
  {
    # direct named filler
    FILTER(isIRI(?filler))
    OPTIONAL { ?filler rdfs:label ?partLabel }
  } UNION {
    # filler is a union — unwrap one level
    ?filler owl:unionOf ?lst .
    ?lst rdf:rest*/rdf:first ?member .
    FILTER(isIRI(?member))
    OPTIONAL { ?member rdfs:label ?partLabel }
  } UNION {
    # filler is an intersection — unwrap one level
    ?filler owl:intersectionOf ?lst .
    ?lst rdf:rest*/rdf:first ?member .
    FILTER(isIRI(?member))
    OPTIONAL { ?member rdfs:label ?partLabel }
  }
} ORDER BY ?flapLabel ?partLabel
""", title="CQ1 TBox — flap classes with compositional has-part restrictions (labels resolved)")


CQ1 TBox — flap classes with compositional has-part restrictions (labels resolved)
  3,986 row(s) (showing first 20)


,flapLabel,partLabel
0,Anterior intercostal artery perforator flap,Anatomical structure
1,Anterior intercostal artery perforator flap,Anterior intercostal artery
2,Anterior intercostal artery perforator flap,Clavicle
3,Anterior intercostal artery perforator flap,Costal cartilage
4,Anterior intercostal artery perforator flap,Inferior epigastric artery
5,Anterior intercostal artery perforator flap,Layer of epidermis proper
6,Anterior intercostal artery perforator flap,Pectoral fascia
7,Anterior intercostal artery perforator flap,Pectoralis major
8,Anterior intercostal artery perforator flap,Posterior intercostal artery
9,Anterior intercostal artery perforator flap,Segment of skin


,flapLabel,partLabel
0,Anterior intercostal artery perforator flap,Anatomical structure
1,Anterior intercostal artery perforator flap,Anterior intercostal artery
2,Anterior intercostal artery perforator flap,Clavicle
3,Anterior intercostal artery perforator flap,Costal cartilage
4,Anterior intercostal artery perforator flap,Inferior epigastric artery
...,...,...
3981,Wrap around flap,Subcutaneous adipose tissue
3982,Wrap around flap,Subdivision of dermis
3983,Wrap around flap,Zone of skin
3984,Z plasty,Anatomical structure


In [31]:
run("""
SELECT ?componentLabel ?typeLabel
WHERE {
  ?flap rdfs:label ?fl . FILTER(CONTAINS(LCASE(?fl), "anterolateral thigh"))
  ?flap ofl:OFLID13296 ?component .
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
} ORDER BY ?componentLabel LIMIT 10
""", title="CQ1 ABox — anatomical components of a generated ALT flap")


CQ1 ABox — anatomical components of a generated ALT flap
  10 row(s)


,componentLabel,typeLabel
0,Flap harvest process of Left Flaps originating...,—
1,Flap harvest process of Left Flaps originating...,—
2,Flap harvest process of Left Flaps originating...,—
3,Flap harvest process of Left Flaps originating...,—
4,Flap harvest process of Left Flaps originating...,—
5,Flap harvest process of Left Flaps originating...,—
6,Flap harvest process of Left Flaps originating...,—
7,Flap harvest process of Left Flaps originating...,—
8,Flap harvest process of Left Flaps originating...,—
9,Flap harvest process of Left Flaps originating...,—


,componentLabel,typeLabel
0,Flap harvest process of Left Flaps originating...,—
1,Flap harvest process of Left Flaps originating...,—
2,Flap harvest process of Left Flaps originating...,—
3,Flap harvest process of Left Flaps originating...,—
4,Flap harvest process of Left Flaps originating...,—
5,Flap harvest process of Left Flaps originating...,—
6,Flap harvest process of Left Flaps originating...,—
7,Flap harvest process of Left Flaps originating...,—
8,Flap harvest process of Left Flaps originating...,—
9,Flap harvest process of Left Flaps originating...,—


In [32]:
run("""
SELECT ?flapLabel ?qualityLabel
WHERE {
  ?flap rdf:type ofl:OFLID10002 .
  ?flap obo:RO_0000086 ?quality .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?quality rdfs:label ?qualityLabel }
} ORDER BY ?flapLabel
""", title="CQ2 ABox — flap individuals with a volume quality")


CQ2 ABox — flap individuals with a volume quality
  19,811 row(s) (showing first 20)


,flapLabel,qualityLabel
0,Alt flap of patient x,Volume of alt flap of patient x
1,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
2,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
3,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
4,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
5,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
6,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
7,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
8,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
9,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...


,flapLabel,qualityLabel
0,Alt flap of patient x,Volume of alt flap of patient x
1,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
2,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
3,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
4,Left Anterior intercostal artery perforator fl...,Volume of Left Anterior intercostal artery per...
...,...,...
19806,Right Wrap around flap of patient 14996,Volume of Right Wrap around flap of patient 14996
19807,Right Wrap around flap of patient 14997,Volume of Right Wrap around flap of patient 14997
19808,Right Wrap around flap of patient 14998,Volume of Right Wrap around flap of patient 14998
19809,Right Wrap around flap of patient 14999,Volume of Right Wrap around flap of patient 14999


In [33]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10135 . FILTER(?t != ofl:OFLID10135)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ3 TBox — all Mathes-Nahai muscle flap types")


CQ3 TBox — all Mathes-Nahai muscle flap types
  16 row(s)


,label
0,Oblique rectus abdominis flap
1,Rectus abdominis flap
2,Rectus abdominis flap with eighth rib
3,Rectus abdominis flap with fascia of rectus ab...
4,Rectus abdominis flap with free upper limb seg...
5,Rectus abdominis flap with ninth rib
6,Rectus abdominis flap with seventh rib
7,Rectus abdominis flap with skin of abdomen
8,Rectus abdominis flap with subcutaneous adipos...
9,Rectus abdominis flap with tenth rib


,label
0,Oblique rectus abdominis flap
1,Rectus abdominis flap
2,Rectus abdominis flap with eighth rib
3,Rectus abdominis flap with fascia of rectus ab...
4,Rectus abdominis flap with free upper limb seg...
5,Rectus abdominis flap with ninth rib
6,Rectus abdominis flap with seventh rib
7,Rectus abdominis flap with skin of abdomen
8,Rectus abdominis flap with subcutaneous adipos...
9,Rectus abdominis flap with tenth rib


In [34]:
run("""
SELECT ?typeLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10135 . FILTER(?t != ofl:OFLID10135)
  OPTIONAL { ?t rdfs:label ?typeLabel }
} GROUP BY ?typeLabel ORDER BY DESC(?n)
""", title="CQ3 ABox — generated individuals by Mathes-Nahai type")


CQ3 ABox — generated individuals by Mathes-Nahai type
  13 row(s)


,typeLabel,n
0,Type iii: two dominant pedicles,860
1,Rectus abdominis flap,860
2,—,860
3,Rectus abdominis flap with eighth rib,100
4,Rectus abdominis flap with tenth rib,100
5,Rectus abdominis flap with skin of abdomen,100
6,Rectus abdominis flap with seventh rib,100
7,Rectus abdominis flap with subcutaneous adipos...,100
8,Rectus abdominis flap with free upper limb seg...,100
9,Rectus abdominis flap with fascia of rectus ab...,100


,typeLabel,n
0,Type iii: two dominant pedicles,860
1,Rectus abdominis flap,860
2,—,860
3,Rectus abdominis flap with eighth rib,100
4,Rectus abdominis flap with tenth rib,100
5,Rectus abdominis flap with skin of abdomen,100
6,Rectus abdominis flap with seventh rib,100
7,Rectus abdominis flap with subcutaneous adipos...,100
8,Rectus abdominis flap with free upper limb seg...,100
9,Rectus abdominis flap with fascia of rectus ab...,100


In [35]:
run("""
SELECT DISTINCT ?flapLabel ?originLabel
WHERE {
  ?flap rdfs:subClassOf* ofl:OFLID10002 .
  ?flap rdfs:subClassOf ?r .
  ?r a owl:Restriction ; owl:onProperty ofl:OFLID12003 ; owl:someValuesFrom ?filler .
  ?flap rdfs:label ?flapLabel .
  {
    FILTER(isIRI(?filler))
    OPTIONAL { ?filler rdfs:label ?originLabel }
  } UNION {
    ?filler owl:unionOf ?lst .
    ?lst rdf:rest*/rdf:first ?member .
    FILTER(isIRI(?member))
    OPTIONAL { ?member rdfs:label ?originLabel }
  }
} ORDER BY ?flapLabel
""", title="CQ4 TBox — flap classes with declared anatomical origin (labels resolved)")


CQ4 TBox — flap classes with declared anatomical origin (labels resolved)
  236 row(s) (showing first 20)


,flapLabel,originLabel
0,Anterior intercostal artery perforator flap,Thorax
1,Anterolateral thigh flap,Segment of thigh
2,Atasoy flap,Finger
3,Auricularis posterior flap,Auriculotemporal part of head
4,Bilateral vy fingertip flap,Finger
5,Deep branch based trapezius flap,Back of thorax
6,Deep inferior epigastric perforator flap,Abdomen
7,Deep inferior epigastric perforator flap with ...,Abdomen
8,Deep inferior epigastric perforator flap with ...,Abdomen
9,Deep inferior epigastric perforator flap with ...,Abdomen


,flapLabel,originLabel
0,Anterior intercostal artery perforator flap,Thorax
1,Anterolateral thigh flap,Segment of thigh
2,Atasoy flap,Finger
3,Auricularis posterior flap,Auriculotemporal part of head
4,Bilateral vy fingertip flap,Finger
...,...,...
231,Vy flaps of the fingertip,Finger
232,Vy flaps of the fingertip flap with finger,Finger
233,Vy flaps of the fingertip flap with skin of fi...,Finger
234,Vy flaps of the fingertip flap with subcutaneo...,Finger


In [36]:
run("""
SELECT ?flapLabel ?originLabel
WHERE {
  ?flap ofl:OFLID12003 ?origin .
  ?flap rdfs:label ?flapLabel .
  OPTIONAL { ?origin rdfs:label ?originLabel }
} ORDER BY ?flapLabel LIMIT 10
""", title="CQ4 ABox — generated individuals with origin assertion (sample)")


CQ4 ABox — generated individuals with origin assertion (sample)
  10 row(s)


,flapLabel,originLabel
0,Left Auricularis posterior flap of patient 301,Auriculotemporal part of head
1,Left Auricularis posterior flap of patient 302,Auriculotemporal part of head
2,Left Auricularis posterior flap of patient 303,Auriculotemporal part of head
3,Left Auricularis posterior flap of patient 304,Auriculotemporal part of head
4,Left Auricularis posterior flap of patient 305,Auriculotemporal part of head
5,Left Auricularis posterior flap of patient 306,Auriculotemporal part of head
6,Left Auricularis posterior flap of patient 307,Auriculotemporal part of head
7,Left Auricularis posterior flap of patient 308,Auriculotemporal part of head
8,Left Auricularis posterior flap of patient 309,Auriculotemporal part of head
9,Left Auricularis posterior flap of patient 310,Auriculotemporal part of head


,flapLabel,originLabel
0,Left Auricularis posterior flap of patient 301,Auriculotemporal part of head
1,Left Auricularis posterior flap of patient 302,Auriculotemporal part of head
2,Left Auricularis posterior flap of patient 303,Auriculotemporal part of head
3,Left Auricularis posterior flap of patient 304,Auriculotemporal part of head
4,Left Auricularis posterior flap of patient 305,Auriculotemporal part of head
5,Left Auricularis posterior flap of patient 306,Auriculotemporal part of head
6,Left Auricularis posterior flap of patient 307,Auriculotemporal part of head
7,Left Auricularis posterior flap of patient 308,Auriculotemporal part of head
8,Left Auricularis posterior flap of patient 309,Auriculotemporal part of head
9,Left Auricularis posterior flap of patient 310,Auriculotemporal part of head


In [37]:
run("""
SELECT ?transferLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap obo:RO_0002488 ?process .
  ?process rdf:type ?procType .
  VALUES ?procType { ofl:OFLID1000137 ofl:OFLID1000138 }
  OPTIONAL { ?procType rdfs:label ?transferLabel }
} GROUP BY ?transferLabel ORDER BY DESC(?n)
""", title="CQ5 ABox — free vs pedicled flap individuals")


CQ5 ABox — free vs pedicled flap individuals
  0 row(s)


,transferLabel,n


,transferLabel,n


In [38]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10086 . FILTER(?t != ofl:OFLID10086)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ6 TBox — distance classification types")


CQ6 TBox — distance classification types
  3 row(s)


,label
0,Distant flaps
1,Local flaps
2,Regional flaps


,label
0,Distant flaps
1,Local flaps
2,Regional flaps


In [39]:
run("""
SELECT ?distLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10086 . FILTER(?t != ofl:OFLID10086)
  OPTIONAL { ?t rdfs:label ?distLabel }
} GROUP BY ?distLabel ORDER BY DESC(?n)
""", title="CQ6 ABox — flap individuals by distance classification")


CQ6 ABox — flap individuals by distance classification
  4 row(s)


,distLabel,n
0,Regional flaps,16950
1,Distant flaps,2830
2,Local flaps,30
3,—,30


,distLabel,n
0,Regional flaps,16950
1,Distant flaps,2830
2,Local flaps,30
3,—,30


In [40]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID1000141 .
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ7 TBox — insertion process subclass hierarchy (partially fulfilled)")


CQ7 TBox — insertion process subclass hierarchy (partially fulfilled)
  1 row(s)


,label
0,Flap insertion process


,label
0,Flap insertion process


In [41]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10131 . FILTER(?t != ofl:OFLID10131)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ8 TBox — pre-harvest modification types")


CQ8 TBox — pre-harvest modification types
  8 row(s)


,label
0,Flaps classified by pre harvest incision
1,Flaps classified by prefabrication
2,Flaps classified by prelamination
3,Flaps preconditioned by delayed harvest
4,Flaps with prefabrication
5,Flaps with prelamination
6,Flaps without prefabrication
7,Flaps without prelamination


,label
0,Flaps classified by pre harvest incision
1,Flaps classified by prefabrication
2,Flaps classified by prelamination
3,Flaps preconditioned by delayed harvest
4,Flaps with prefabrication
5,Flaps with prelamination
6,Flaps without prefabrication
7,Flaps without prelamination


In [42]:
run("""
SELECT ?modLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10045 . FILTER(?t != ofl:OFLID10045)
  OPTIONAL { ?t rdfs:label ?modLabel }
} GROUP BY ?modLabel ORDER BY DESC(?n)
""", title="CQ8 ABox — pre-harvest modification counts")


CQ8 ABox — pre-harvest modification counts
  2 row(s)


,modLabel,n
0,Flaps without preharvest modification,10160
1,Flaps with preharvest modification,9650


,modLabel,n
0,Flaps without preharvest modification,10160
1,Flaps with preharvest modification,9650


In [43]:
run("""
SELECT ?label ?citation
WHERE {
  OPTIONAL { ofl:OFLID1000185 rdfs:label ?label }
  OPTIONAL { ofl:OFLID1000185 dcterms:bibliographicCitation ?citation }
}
""", title="CQ9 — STSG concept in TBox (not yet linked to donor-site flaps)")


CQ9 — STSG concept in TBox (not yet linked to donor-site flaps)
  1 row(s)


,label,citation
0,Split thickness skin graft,—


,label,citation
0,Split thickness skin graft,—


In [44]:
run("""
SELECT DISTINCT ?label
WHERE {
  ?t rdfs:subClassOf* ofl:OFLID10177 . FILTER(?t != ofl:OFLID10177)
  OPTIONAL { ?t rdfs:label ?label }
} ORDER BY ?label
""", title="CQ10 TBox — survival outcome classes")


CQ10 TBox — survival outcome classes
  9 row(s)


,label
0,Flap with loss due to infection
1,Flaps classified by cause of flap loss
2,Flaps classified by type of partial loss
3,Flaps with loss at the apex
4,Flaps with loss due to arterial obstruction
5,Flaps with loss due to venous obstruction
6,Flaps with tissue loss
7,Flaps with total loss
8,Flaps without tissue loss


,label
0,Flap with loss due to infection
1,Flaps classified by cause of flap loss
2,Flaps classified by type of partial loss
3,Flaps with loss at the apex
4,Flaps with loss due to arterial obstruction
5,Flaps with loss due to venous obstruction
6,Flaps with tissue loss
7,Flaps with total loss
8,Flaps without tissue loss


In [45]:
run("""
SELECT ?outcomeLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?t .
  ?t rdfs:subClassOf* ofl:OFLID10177 . FILTER(?t != ofl:OFLID10177)
  OPTIONAL { ?t rdfs:label ?outcomeLabel }
} GROUP BY ?outcomeLabel ORDER BY DESC(?n)
""", title="CQ10 ABox — survival outcome counts")


CQ10 ABox — survival outcome counts
  8 row(s)


,outcomeLabel,n
0,Flaps with tissue loss,15780
1,Flaps classified by cause of flap loss,7720
2,Flaps without tissue loss,4030
3,Flaps classified by type of partial loss,4030
4,Flaps with loss at the apex,4030
5,Flaps with total loss,4030
6,Flaps with loss due to venous obstruction,3860
7,Flaps with loss due to arterial obstruction,3860


,outcomeLabel,n
0,Flaps with tissue loss,15780
1,Flaps classified by cause of flap loss,7720
2,Flaps without tissue loss,4030
3,Flaps classified by type of partial loss,4030
4,Flaps with loss at the apex,4030
5,Flaps with total loss,4030
6,Flaps with loss due to venous obstruction,3860
7,Flaps with loss due to arterial obstruction,3860


In [46]:
print("\n" + "="*60)
print("Grand totals")
n_flaps = list(g.query(PREFIXES + "SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?f rdf:type ofl:OFLID10002 . }"))[0][0]
n_ind   = sum(1 for _ in g.subjects(RDF.type, OWL.NamedIndividual))
print(f"  Flap individuals (direct type ofl:OFLID10002) : {n_flaps}")
print(f"  All named individuals                         : {n_ind:,}")
print(f"  Total triples (TBox + ABox + FMA)             : {len(g):,}")


Grand totals
  Flap individuals (direct type ofl:OFLID10002) : 20111
  All named individuals                         : 203,038
  Total triples (TBox + ABox + FMA)             : 11,658,385



---
## Diagnostic Queries

Investigating logical inconsistencies revealed by the reasoner: unsatisfiable classes and malformed built-in concepts.


In [9]:
# Find ALL classes that directly have owl:Nothing as a superclass
# NOTE: SPARQL property paths (rdfs:subClassOf*) may not work in rdflib
# This query finds DIRECT subClassOf statements pointing to owl:Nothing
run("""
SELECT DISTINCT ?classLabel ?classIRI
WHERE {
  ?class rdfs:subClassOf owl:Nothing .
  BIND(STR(?class) AS ?classIRI)
  OPTIONAL { ?class rdfs:label ?classLabel }
} ORDER BY ?classLabel
""", title="DIAGNOSTIC — All classes directly declaring owl:Nothing as superclass", limit=100, graph=g_tbox)



DIAGNOSTIC — All classes directly declaring owl:Nothing as superclass
  1 row(s)


,classLabel,classIRI
0,—,http://www.w3.org/2002/07/owl#Nothing


,classLabel,classIRI
0,—,http://www.w3.org/2002/07/owl#Nothing


In [8]:
# Inspect the definitions of owl:Nothing and owl:Thing
run("""
SELECT ?entity ?predicate ?object
WHERE {
  VALUES ?entity { owl:Nothing owl:Thing }
  ?entity ?predicate ?object .
  FILTER(?predicate != rdf:type)
} ORDER BY ?entity ?predicate
""", title="Definitions of owl:Nothing and owl:Thing in the inferred ontology", limit=50, graph=g_tbox)



Definitions of owl:Nothing and owl:Thing in the inferred ontology
  3,394 row(s) (showing first 50)


,entity,predicate,object
0,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,N82b516992ecb419cb5c38f25f10a74ce
1,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,Nd777d09a6fa745ad865d2fc11e664b4b
2,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...
3,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,http://purl.obolibrary.org/obo/FMA_57815
4,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...
5,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,http://purl.obolibrary.org/obo/FMA_74941
6,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...
7,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...
8,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...
9,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...


,entity,predicate,object
0,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,N82b516992ecb419cb5c38f25f10a74ce
1,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,Nd777d09a6fa745ad865d2fc11e664b4b
2,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...
3,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,http://purl.obolibrary.org/obo/FMA_57815
4,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2000/01/rdf-schema#subClassOf,https://purl.bioontology.org/ontology/OFL/OFLI...
...,...,...,...
3389,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2002/07/owl#equivalentClass,http://www.w3.org/2002/07/owl#Nothing
3390,http://www.w3.org/2002/07/owl#Nothing,http://www.w3.org/2002/07/owl#sameAs,http://www.w3.org/2002/07/owl#Nothing
3391,http://www.w3.org/2002/07/owl#Thing,http://www.w3.org/2000/01/rdf-schema#subClassOf,http://www.w3.org/2002/07/owl#Thing
3392,http://www.w3.org/2002/07/owl#Thing,http://www.w3.org/2002/07/owl#equivalentClass,http://www.w3.org/2002/07/owl#Thing


### Explanation: owl:Thing Equivalence Symbol

**Why does `owl:Thing` display an equivalence symbol (≡) but has no explicit `owl:equivalentClass` axiom?**

1. **`owl:Thing` is built-in**: The OWL specification defines `owl:Thing` as a universal concept (the top of the class hierarchy). It doesn't need an axiom.

2. **Protégé/Reasoner display**: The "≡" symbol in Protégé appears when a class is:
   - Inferred to be equivalent to another class through logical reasoning
   - OR simply to indicate its special status as a built-in class

3. **What should be true** (in correct OWL):
   - Every user-defined class should have `rdfs:subClassOf owl:Thing`
   - `owl:Nothing` should have `rdfs:subClassOf owl:Thing` (it's the empty class—still under Thing)
   - `owl:Nothing` should NOT have superclasses pointing to user classes

4. **What's wrong in your inferred file**:
   - `owl:Nothing` has user-defined classes as superclasses (reversed!)
   - This suggests classes are being inferred as **unsatisfiable** (contradictory constraints)
   - The reasoner is saying: "This class definition is logically impossible, so only `owl:Nothing` can satisfy it"

**Bottom line**: The ≡ symbol on `owl:Thing` is cosmetic/informational. The real problem is that `owl:Nothing` has the hierarchy backwards—it should be at the **bottom**, not with user classes above it.


In [11]:
print("\n" + "="*60)
print("Diagnostic: owl:Nothing definition in inferred file")
print("="*60)

from rdflib import Graph
g_raw = Graph()
g_raw.parse(TBOX, format="xml")

owl_nothing = URIRef("http://www.w3.org/2002/07/owl#Nothing")

# Count all triples where owl:Nothing is subject
total_triples_nothing_subject = len(list(g_raw.triples((owl_nothing, None, None))))
print(f"\nTotal triples where owl:Nothing is subject: {total_triples_nothing_subject}")

# Count rdfs:subClassOf where subject is owl:Nothing
subclass_of_where_nothing_subject = list(g_raw.objects(owl_nothing, RDFS.subClassOf))
print(f"rdfs:subClassOf statements where owl:Nothing is subject: {len(subclass_of_where_nothing_subject)}")

# Count classes that are rdfs:subClassOf owl:Nothing
classes_subclasses_of_nothing = list(g_raw.subjects(RDFS.subClassOf, owl_nothing))
print(f"Classes declaring rdfs:subClassOf owl:Nothing: {len(classes_subclasses_of_nothing)}")

# Show examples (excluding owl:Nothing itself)
print(f"\nExamples of classes with rdfs:subClassOf owl:Nothing (excluding owl:Nothing):")
count = 0
for s in classes_subclasses_of_nothing:
    if str(s) != str(owl_nothing) and count < 10:
        label = list(g_raw.objects(s, RDFS.label))
        label_str = label[0] if label else "(no label)"
        print(f"  {count+1}. {label_str}")
        print(f"     IRI: {s}")
        count += 1

if len(classes_subclasses_of_nothing) > 11:
    print(f"  ... and {len(classes_subclasses_of_nothing) - 10} more")

print(f"\n⚠️  Total problematic classes: {len([s for s in classes_subclasses_of_nothing if str(s) != str(owl_nothing)])}")
print("   These classes are UNSATISFIABLE (logically contradictory).")



Diagnostic: owl:Nothing definition in inferred file

Total triples where owl:Nothing is subject: 3392
rdfs:subClassOf statements where owl:Nothing is subject: 3389
Classes declaring rdfs:subClassOf owl:Nothing: 1

Examples of classes with rdfs:subClassOf owl:Nothing (excluding owl:Nothing):

⚠️  Total problematic classes: 0
   These classes are UNSATISFIABLE (logically contradictory).


In [12]:
print("\n" + "="*60)
print("The ONE unsatisfiable class:")
print("="*60)

# Find the specific class that is rdfs:subClassOf owl:Nothing
for s in g_raw.subjects(RDFS.subClassOf, owl_nothing):
    if str(s) != str(owl_nothing):
        labels = list(g_raw.objects(s, RDFS.label))
        print(f"\nClass IRI: {s}")
        if labels:
            print(f"Label: {labels[0]}")
        else:
            print("(No label)")
        
        # Show all its superclasses to understand why it's unsatisfiable
        superclasses = list(g_raw.objects(s, RDFS.subClassOf))
        print(f"\nThis class declares {len(superclasses)} rdfs:subClassOf statements:")
        print("(First 15 superclasses shown)")
        for i, sup in enumerate(superclasses[:15]):
            sup_labels = list(g_raw.objects(sup, RDFS.label))
            sup_label = sup_labels[0] if sup_labels else "(no label)"
            print(f"  {i+1}. {sup_label}")
        if len(superclasses) > 15:
            print(f"  ... and {len(superclasses) - 15} more")



The ONE unsatisfiable class:


---
## Task-Based Evaluation

Five realistic clinical and research tasks demonstrating the ontology supports end-to-end use cases beyond abstract CQ answering.

### Task 1 — Flap selection by donor region
**Clinical scenario:** A surgeon needs a pedicled flap for trunk reconstruction and wants to know which flap types originate from the back of the trunk and are transferred regionally.

*Query: find all flap classes whose `has-flap-origin` restriction points to "Back of trunk", then count how many ABox individuals of those classes are pedicled.*

In [47]:
# Task 1a — TBox: which flap classes originate from the back of the trunk?
run("""
SELECT DISTINCT ?flapLabel ?originLabel
WHERE {
  ?flap rdfs:subClassOf* ofl:OFLID10002 .
  ?flap rdfs:subClassOf ?r .
  ?r a owl:Restriction ; owl:onProperty ofl:OFLID12003 ; owl:someValuesFrom ?origin .
  ?origin rdfs:label ?originLabel .
  FILTER(CONTAINS(LCASE(?originLabel), "back of trunk"))
  ?flap rdfs:label ?flapLabel .
} ORDER BY ?flapLabel
""", title="Task 1a — Flap classes with origin in 'Back of trunk'")


Task 1a — Flap classes with origin in 'Back of trunk'
  17 row(s)


,flapLabel,originLabel
0,Lateral thoracic perforator flap,Back of trunk
1,Lateral thoracic perforator flap with skin of ...,Back of trunk
2,Lateral thoracic perforator flap with subcutan...,Back of trunk
3,Lateral thoracic perforator flap with superfic...,Back of trunk
4,Latissimus dorsi flap,Back of trunk
5,Latissimus dorsi flap with fascia of latissimu...,Back of trunk
6,Latissimus dorsi flap with fascia of serratus ...,Back of trunk
7,Latissimus dorsi flap with rib,Back of trunk
8,Latissimus dorsi flap with scapula,Back of trunk
9,Latissimus dorsi flap with serratus anterior,Back of trunk


,flapLabel,originLabel
0,Lateral thoracic perforator flap,Back of trunk
1,Lateral thoracic perforator flap with skin of ...,Back of trunk
2,Lateral thoracic perforator flap with subcutan...,Back of trunk
3,Lateral thoracic perforator flap with superfic...,Back of trunk
4,Latissimus dorsi flap,Back of trunk
5,Latissimus dorsi flap with fascia of latissimu...,Back of trunk
6,Latissimus dorsi flap with fascia of serratus ...,Back of trunk
7,Latissimus dorsi flap with rib,Back of trunk
8,Latissimus dorsi flap with scapula,Back of trunk
9,Latissimus dorsi flap with serratus anterior,Back of trunk


In [48]:
# Task 1b — ABox: pedicled instances of those flap types with their survival
run("""
SELECT ?flapLabel ?survivalLabel
WHERE {
  ?flap ofl:OFLID12003 ?origin .
  ?origin rdfs:label ?originLabel .
  FILTER(CONTAINS(LCASE(?originLabel), "back of trunk"))
  ?flap obo:RO_0002488 ?process .
  ?process rdf:type ofl:OFLID1000138 .   # pedicled
  ?flap rdf:type ?survCls .
  ?survCls rdfs:subClassOf* ofl:OFLID10177 .
  FILTER(?survCls != ofl:OFLID10177)
  ?flap rdfs:label ?flapLabel .
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} ORDER BY ?flapLabel LIMIT 15
""", title="Task 1b — Pedicled back-of-trunk flap individuals with survival outcome (sample)")


Task 1b — Pedicled back-of-trunk flap individuals with survival outcome (sample)
  0 row(s)


,flapLabel,survivalLabel


,flapLabel,survivalLabel


### Task 2 — Free flap survival audit
**Research scenario:** A clinical researcher wants to audit all free flap procedures and produce a survival distribution table — the kind of result reported in outcome studies.

*Query: for every free flap individual, retrieve its survival classification and aggregate counts.*

In [49]:
run("""
SELECT ?survivalLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap obo:RO_0002488 ?process .
  ?process rdf:type ofl:OFLID1000137 .     # free transfer
  ?flap rdf:type ?survCls .
  ?survCls rdfs:subClassOf* ofl:OFLID10177 .
  FILTER(?survCls != ofl:OFLID10177)
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} GROUP BY ?survivalLabel ORDER BY DESC(?n)
""", title="Task 2 — Survival distribution across all free flap individuals")


Task 2 — Survival distribution across all free flap individuals
  0 row(s)


,survivalLabel,n


,survivalLabel,n


### Task 3 — Mathes-Nahai Type I flap lookup
**Educational scenario:** A resident wants to know which muscle flaps have a single dominant vascular pedicle (Mathes-Nahai Type I) and how many cases of each are represented.

*Query: retrieve all flap classes that are subclasses of Type I, and count ABox individuals per type.*

In [50]:
# TBox: for each Mathes-Nahai type, which named flap classes are subclasses of it?
run("""
SELECT DISTINCT ?mnTypeLabel ?flapLabel
WHERE {
  ?mnType rdfs:subClassOf ofl:OFLID10135 .    # direct children of Mathes-Nahai root
  ?flap rdfs:subClassOf* ?mnType .
  FILTER(?flap != ?mnType)
  FILTER(STRSTARTS(STR(?flap), "https://purl.bioontology.org/ontology/OFL/"))
  ?mnType rdfs:label ?mnTypeLabel .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
} ORDER BY ?mnTypeLabel ?flapLabel
""", title="Task 3a — Named flap classes per Mathes-Nahai vascular pattern type")


Task 3a — Named flap classes per Mathes-Nahai vascular pattern type
  37 row(s) (showing first 20)


,mnTypeLabel,flapLabel
0,Muscle flaps classified by mathes and nahai,Oblique rectus abdominis flap
1,Muscle flaps classified by mathes and nahai,Rectus abdominis flap
2,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with eighth rib
3,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with fascia of rectus ab...
4,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with free upper limb seg...
5,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with ninth rib
6,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with seventh rib
7,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with skin of abdomen
8,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with subcutaneous adipos...
9,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with tenth rib


,mnTypeLabel,flapLabel
0,Muscle flaps classified by mathes and nahai,Oblique rectus abdominis flap
1,Muscle flaps classified by mathes and nahai,Rectus abdominis flap
2,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with eighth rib
3,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with fascia of rectus ab...
4,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with free upper limb seg...
5,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with ninth rib
6,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with seventh rib
7,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with skin of abdomen
8,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with subcutaneous adipos...
9,Muscle flaps classified by mathes and nahai,Rectus abdominis flap with tenth rib


In [51]:
# ABox: count instances per Mathes-Nahai type (flap individuals typed to subclasses of OFLID10135)
run("""
SELECT ?mnTypeLabel ?flapTypeLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ?flapType .
  ?flapType rdfs:subClassOf* ?mnType .
  ?mnType rdfs:subClassOf ofl:OFLID10135 .
  OPTIONAL { ?mnType rdfs:label ?mnTypeLabel }
  OPTIONAL { ?flapType rdfs:label ?flapTypeLabel }
} GROUP BY ?mnTypeLabel ?flapTypeLabel ORDER BY ?mnTypeLabel DESC(?n)
""", title="Task 3b — ABox instance counts per Mathes-Nahai type and flap class")


Task 3b — ABox instance counts per Mathes-Nahai type and flap class
  70 row(s) (showing first 20)


,mnTypeLabel,flapTypeLabel,n
0,—,—,860
1,—,Type iii: two dominant pedicles,860
2,—,Rectus abdominis flap,860
3,—,Rectus abdominis flap with eighth rib,100
4,—,Rectus abdominis flap with tenth rib,100
5,—,Rectus abdominis flap with fascia of rectus ab...,100
6,—,Rectus abdominis flap with subcutaneous adipos...,100
7,—,Rectus abdominis flap with ninth rib,100
8,—,Rectus abdominis flap with free upper limb seg...,100
9,—,Rectus abdominis flap with skin of abdomen,100


,mnTypeLabel,flapTypeLabel,n
0,—,—,860
1,—,Type iii: two dominant pedicles,860
2,—,Rectus abdominis flap,860
3,—,Rectus abdominis flap with eighth rib,100
4,—,Rectus abdominis flap with tenth rib,100
...,...,...,...
65,Type iii: two dominant pedicles,Rectus abdominis flap with fascia of rectus ab...,100
66,Type iii: two dominant pedicles,Rectus abdominis flap with ninth rib,100
67,Type iii: two dominant pedicles,Oblique rectus abdominis flap,30
68,Type iii: two dominant pedicles,Vertical rectus abdominis flap,30


### Task 4 — Prefabrication outcome tracking
**Research scenario:** A researcher investigates whether pre-harvest modification affects flap survival. Query all prefabricated flap individuals and cross-tabulate with survival outcome.

In [52]:
run("""
SELECT ?survivalLabel (COUNT(DISTINCT ?flap) AS ?n)
WHERE {
  ?flap rdf:type ofl:OFLID10131 .          # flaps with preharvest modification
  ?flap rdf:type ?survCls .
  ?survCls rdfs:subClassOf* ofl:OFLID10177 .
  FILTER(?survCls != ofl:OFLID10177)
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} GROUP BY ?survivalLabel ORDER BY DESC(?n)
""", title="Task 4 — Survival outcomes for pre-harvest modified flaps")


Task 4 — Survival outcomes for pre-harvest modified flaps
  8 row(s)


,survivalLabel,n
0,Flaps with tissue loss,7720
1,Flaps classified by cause of flap loss,3860
2,Flaps with total loss,1930
3,Flaps with loss due to arterial obstruction,1930
4,Flaps classified by type of partial loss,1930
5,Flaps with loss at the apex,1930
6,Flaps with loss due to venous obstruction,1930
7,Flaps without tissue loss,1930


,survivalLabel,n
0,Flaps with tissue loss,7720
1,Flaps classified by cause of flap loss,3860
2,Flaps with total loss,1930
3,Flaps with loss due to arterial obstruction,1930
4,Flaps classified by type of partial loss,1930
5,Flaps with loss at the apex,1930
6,Flaps with loss due to venous obstruction,1930
7,Flaps without tissue loss,1930


### Task 5 — Operative planning: anatomical component inventory
**Clinical scenario:** Before harvesting a Fibula flap, the surgeon wants a complete list of anatomical structures defined in the ontology as components of this flap — directly queryable from the TBox without needing a textbook.

In [53]:
# TBox: has-part restrictions on Fibula flap — resolve union/intersection fillers via ABox component types
# Since Fibula flap fillers are union blank nodes, we query the ABox component individuals
# and retrieve their FMA class labels (same information, grounded in instances)
run("""
SELECT DISTINCT ?typeLabel (COUNT(DISTINCT ?component) AS ?n)
WHERE {
  ?flap rdfs:label ?fl . FILTER(CONTAINS(LCASE(?fl), "fibula flap"))
  ?flap ofl:OFLID13296 ?component .
  ?component rdf:type ?t .
  FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
  OPTIONAL { ?t rdfs:label ?typeLabel }
} GROUP BY ?typeLabel ORDER BY ?typeLabel
""", title="Task 5 — Distinct FMA anatomical types harvested in Fibula flap instances")


Task 5 — Distinct FMA anatomical types harvested in Fibula flap instances
  10 row(s)


,typeLabel,n
0,Fascia of flexor hallucis longus,100
1,Fascia of soleus,100
2,Flexor hallucis longus,100
3,Posterior intermuscular septum of leg,100
4,Skin of leg,100
5,Small saphenous vein,100
6,Soleus,100
7,Subcutaneous adipose tissue,100
8,Superficial fascia of leg,100
9,Tibialis posterior,100


,typeLabel,n
0,Fascia of flexor hallucis longus,100
1,Fascia of soleus,100
2,Flexor hallucis longus,100
3,Posterior intermuscular septum of leg,100
4,Skin of leg,100
5,Small saphenous vein,100
6,Soleus,100
7,Subcutaneous adipose tissue,100
8,Superficial fascia of leg,100
9,Tibialis posterior,100


In [54]:
# ABox: actual component individuals of a generated Fibula flap instance
run("""
SELECT ?componentLabel ?typeLabel
WHERE {
  ?flap rdfs:label ?fl . FILTER(CONTAINS(LCASE(?fl), "fibula flap"))
  ?flap ofl:OFLID13296 ?component .
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
} ORDER BY ?componentLabel LIMIT 12
""", title="Task 5 — Component individuals of a generated Fibula flap ABox instance")


Task 5 — Component individuals of a generated Fibula flap ABox instance
  12 row(s)


,componentLabel,typeLabel
0,Flap harvest process of Left Fibula flap with ...,—
1,Flap harvest process of Left Fibula flap with ...,—
2,Flap harvest process of Left Fibula flap with ...,—
3,Flap harvest process of Left Fibula flap with ...,—
4,Flap harvest process of Left Fibula flap with ...,—
5,Flap harvest process of Left Fibula flap with ...,—
6,Flap harvest process of Left Fibula flap with ...,—
7,Flap harvest process of Left Fibula flap with ...,—
8,Flap harvest process of Left Fibula flap with ...,—
9,Flap harvest process of Left Fibula flap with ...,—


,componentLabel,typeLabel
0,Flap harvest process of Left Fibula flap with ...,—
1,Flap harvest process of Left Fibula flap with ...,—
2,Flap harvest process of Left Fibula flap with ...,—
3,Flap harvest process of Left Fibula flap with ...,—
4,Flap harvest process of Left Fibula flap with ...,—
5,Flap harvest process of Left Fibula flap with ...,—
6,Flap harvest process of Left Fibula flap with ...,—
7,Flap harvest process of Left Fibula flap with ...,—
8,Flap harvest process of Left Fibula flap with ...,—
9,Flap harvest process of Left Fibula flap with ...,—
